# 🎯 [TRN02_M08] 전 부대원 동시 응시 모의 대규모 트래픽 채점 부하 테스트 및 최적화 (학생 실습용)
### Google Colab 실습 환경 (교육·훈련 / Level 2 - 학생 실습용)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/edunlab/milicontent2/blob/main/uploads/TRN02/trn02_m08_student.ipynb)

---

## 📌 1. 프로젝트 및 미션 개요

| 항목 | 내용 |
| :--- | :--- |
| **과정 분류** | 교육·훈련 (Level 2) |
| **미션 ID** | TRN02_M08 |
| **미션 주제** | 전 부대원 동시 응시 모의 대규모 트래픽 채점 부하 테스트 및 최적화 |
| **소요 시간** | 약 3.5시간 |
| **환경 사양** | Python 3.10+, numpy, pandas, matplotlib, scipy, scikit-learn |

### 📖 프로젝트 목표
미국-이란 전쟁 사례에서 보듯 드론과 미사일이 섞여 날아오는 전장에서는 장병 한 사람 한 사람이 비행체 형상을 식별하고, 아군·민간기와 구분하며, 정해진 보고·통제 절차를 지키는 능력이 방공의 기초가 된다. 그러나 부대 단위 방공 식별 교육은 채점과 오답 분석에 교관의 손이 많이 들어 개인별 약점을 제때 짚어 주기 어렵다. 본 프로젝트에서는 교육용 가상 문항 은행과 가상 응시자 응답을 코드로 만들어, 문항 데이터베이스 정의, 응답 정제·파싱, 자동 채점, 오답 유형(개념 혼동·피아 오인·절차 누락) 분류, 취약 영역 추천, 교수자 대시보드, 교관 검토(HITL), 대규모 동시 응시 부하 시험, AAR 문항 분석과 재보정, 최종 통합까지 채점·피드백 체계를 단계별로 구현한다. 최종 목표는 채점은 빠르고 정확하게 자동화하되, 이의 제기와 서술형 판단은 교관이 확정하는 신뢰할 수 있는 방공 교육 평가 도구를 만드는 것이다.

> **⚠️ [교육용 가상 데이터셋 원칙]**
> - 본 노트북의 모든 데이터(제원, 로그, 시나리오 등)는 보안 규정을 준수하여 **코드 내 난수 기반 가상 데이터로 생성**됩니다.
> - 실제 부대 위치, 작전 인가, 실제 전장 정보, 군사 기밀 데이터는 일체 포함되지 않습니다.

---

## 📂 2. 산출물 디렉토리 체계 및 실습 가이드

본 과제의 미션별 산출물(`deliverables`)은 단순 스크립트를 넘어 실무 엔터프라이즈 및 국방 교육·훈련 평가체계에 즉시 배포할 수 있는 **표준 패키지 디렉토리 구조**를 따릅니다:

| 디렉토리 | 주요 산출물(Deliverables) | 역할 및 설명 |
| :--- | :--- | :--- |
| **`data/`** | `data/schema/trn02_input_schema.json` | 문항 데이터베이스 스키마 및 가상 응답 데이터셋 규격 |
| **`src/`** | `src/preprocess/`, `src/models/`, `src/verification/`, `src/ui/`, `src/hitl/`, `src/main_trn02.py` | 응답 전처리, 자동 채점 Baseline, 오답 유형 분류, 취약 영역 추천, 교수자 대시보드, HITL 등 핵심 파이썬 소스 코드 |
| **`docs/`** | `docs/trn02_data_spec.md`, `docs/trn02_hitl_guide.md`, `docs/trn02_aar_report.md`, `docs/trn02_final_report.md` | 데이터 규격서, HITL 교관 검토 가이드, AAR 분석 보고서, 최종 운영 매뉴얼 등 기술/행정 문서 |
| **`configs/`**| `configs/trn02_tuned_params.yaml` | AAR 피드백 기반 재조정 변별도 파라미터 및 가중치 설정 파일 |
| **`tests/`** | `tests/trn02_stress_test.py` | 전 부대원 동시 응시 시나리오 대규모 트래픽 부하 테스트 및 최적화 스위트 |

> **💡 [Google Colab / 주피터 노트북 실습 팁]**
> - **방법 A. 자동 파일 저장 (권장):** 코드 셀 첫 줄에 `%%writefile src/...`와 같이 매직 명령어를 추가하여 실행하면, 노트북에서 작업한 코드가 해당 폴더의 파일로 자동 추출·저장됩니다.
> - **방법 B. 논리적 모듈 대응:** 별도 파일 추출 없이 노트북 셀 단위로 순차 실행하더라도 각 미션이 해당 모듈의 역할을 1:1로 수행하며, **미션 10(시스템 통합)**에서 전체 파이프라인이 유기적으로 완성되도록 설계되어 있습니다.

---


In [ ]:
# ============================================================================
# [Google Colab 환경 설정] 필수 라이브러리 설치 및 임포트
# ============================================================================
!pip install -q numpy pandas matplotlib scipy scikit-learn

import os
import sys
import time
import math
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# 그래프 영문 폰트 기본 설정 (환경 독립성 보장)
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

print(f"실행 환경 준비 완료 (Python {sys.version.split()[0]})")


---

## 🎯 미션 08. 전 부대원 동시 응시 모의 대규모 트래픽 채점 부하 테스트 및 최적화
**소요시간**: 약 3.5시간

### 📋 미션 배경 및 수행 지침
#### 미션 배경 및 당위성

전 부대원 동시 응시 모의 대규모 트래픽 채점 부하 테스트를 수행한다. 실제 전술 현장에서는 단순 탐지에 그치지 않고 신속한 위협 판단과 신뢰도 높은 의사결정이 필수적이므로, **전 부대원이 동시에 응시하는 상황을 모의해 채점 시스템이 대규모 제출과 장애 속에서도 멈추거나 답안을 잃지 않는지 시험한다.** 과업을 완수하여 오판을 방지하고 작전 대응력을 극대화해야 합니다.

이 문제를 해결하기 위해 본 미션에서는 **평시 120명 제출량을 3배·6배·11배로 늘리고 초장문·제어문자·없는 문항 등 손상 답안과 채점기 응답 지연(장애 2%) 주입하여 처리 지연 및 복원력을 시험하는** 실습을 체계적으로 수행합니다.

학습자는 실습가이드에 제시된 총 4개 세부 활동(기본 기능 구현 ➔ 결과 지표 검증 ➔ 파라미터 튜닝 및 도전과제)을 순차적으로 진행하며 핵심 판정 기준을 달성합니다.

**이번 결과물:** `tests/trn02_stress_test.py 및 스트레스 테스트 분석 보고서` 형태의 결과물을 완성한다.

### 🔍 평가 및 검증 기준 (PASS 조건)
#### 미션 판정

**평가 구분:** 필수

**핵심 평가영역:** 극단 상황에서의 안전 실패(Fail-safe) 및 기능 축소 작동 적절성

#### 미션 PASS 조건

- 정량 기준: 동시 트래픽 500% 증가 시 시스템 다운 0건, 데이터 유실률 0%


- 진행 기준: 대량 시뮬레이션 부하 주입 테스트 통과 및 병목 구간 프로파일링 완료


- 검증 방법: 부하 테스트 도구를 통한 처리량(Throughput) 및 메모리 누수 검증


#### 채점자 판정 메모

> 단순 코드 실행 여부뿐만 아니라 출력된 수치와 근거 자료가 군 실무 요구사항에 부합하는지 면밀히 확인한다.



In [ ]:
"""전 부대원 동시 응시를 모의해 평시(120명) 대비 최대 1,000% 늘어난 제출량과 손상 답안·채점기 장애를 주입하는 채점 부하 테스트를 수행한다. 처리량, 시스템 다운, 데이터 유실을 측정하고, 장애 시 '채점 보류'로 보존하는 안전 실패 동작과 캐시 최적화 전후 처리량을 비교한다."""
import difflib
import functools
import re
import time
import tracemalloc
import unicodedata
from collections import deque
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ---------------------------------------------------------------------------
# [교육용가상자료] 방공·드론 식별 퀴즈 문항 은행과 가상 응시자 응답 생성기
# 문항은 일반 원칙 수준의 가상 문항이며 실제 교범·교전규칙 원문이 아니다. 응시자는 가명(S001~)이다.
# ---------------------------------------------------------------------------
AREAS = ["형상식별", "피아식별", "경보보고", "교전통제", "대드론일반"]
ETYPES = ["개념 혼동", "피아 오인", "절차 누락"]
POINTS = {1: 2, 2: 3, 3: 5}
C, F, P = ETYPES


def _mc(qid, area, diff, stem, opts, ans, dist):
    # [TODO : 학생 실습 8] 전 부대원 동시 응시 모의 대규모 트래픽 채점 부하 테스트 및 최적화 관련 핵심 로직을 구현하세요.
    # 힌트: 상단 주석 및 데이터 스키마/지침을 참고하여 작성합니다.
    # <<< 여기에 코드를 작성하세요 >>>
    pass  # 작성 후 pass를 제거하세요.
def _sa(qid, area, diff, stem, accept, wrong):
    # [TODO : 학생 실습 8] 전 부대원 동시 응시 모의 대규모 트래픽 채점 부하 테스트 및 최적화 관련 핵심 로직을 구현하세요.
    # 힌트: 상단 주석 및 데이터 스키마/지침을 참고하여 작성합니다.
    # <<< 여기에 코드를 작성하세요 >>>
    pass  # 작성 후 pass를 제거하세요.
def _order(qid, area, diff, stem):
    return {"qid": qid, "type": "ORDER", "area": area, "diff": diff, "stem": stem, "answer": "ABCD"}


BANK = [
    _mc("Q01", "형상식별", 1, "꼬리날개 없이 삼각형 날개를 가진 소형 고정익 실루엣은?", ["쿼드콥터형", "삼각익 고정익형", "회전익 헬기형", "조류"], 2, [C, C, F]),
    _mc("Q02", "형상식별", 1, "회전날개 4개가 십자로 배치된 실루엣은?", ["고정익 드론형", "쿼드콥터형", "순항미사일형", "탄도미사일형"], 2, [C, C, C]),
    _mc("Q03", "형상식별", 2, "가늘고 긴 동체 중간에 작은 날개가 붙은 실루엣은?", ["순항미사일형", "탄도미사일형", "민간 경비행기형", "조류"], 1, [C, F, F]),
    _mc("Q04", "형상식별", 3, "원거리라 실루엣 식별이 어려울 때 먼저 할 행동은?", ["추정 유형으로 확정 보고", "관측 정보를 그대로 보고하고 추가 관측", "관측 중단", "민간기로 간주"], 2, [P, P, F]),
    _mc("Q05", "피아식별", 2, "식별 신호 응답이 없는 항공기에 대한 올바른 판단은?", ["즉시 적기로 확정", "미확인 항공기로 분류하고 추가 확인", "아군기로 간주", "무시"], 2, [F, F, P]),
    _mc("Q06", "피아식별", 2, "운항 정보와 일치하는 저속 비행체의 분류는?", ["적 드론", "민간기 가능성 높음, 추가 확인", "순항미사일", "아군 정찰기로 확정"], 2, [F, C, F]),
    _mc("Q07", "피아식별", 3, "아군 드론 운용 공지가 있는 공역에서 미확인 드론을 보면 우선 확인할 것은?", ["기상 정보", "아군 운용 계획과 식별 신호", "대응 준비 상태", "주변 지형"], 2, [C, P, C]),
    _mc("Q08", "피아식별", 3, "조류 무리와 소형 드론 군집 구분에 가장 도움이 되는 정보는?", ["크기만 비교", "궤적의 규칙성과 열 신호", "색상", "관측 시각"], 2, [C, C, C]),
    _mc("Q09", "경보보고", 1, "공중 위협 최초 보고에 포함하지 않아도 되는 것은?", ["관측 시각", "방향과 거리", "추정 유형", "관측자 개인 연락처"], 4, [P, P, P]),
    _mc("Q10", "경보보고", 2, "경보 전파 후 상황이 바뀌면?", ["최초 보고 유지", "변경 사항을 즉시 추가 보고", "상황 종료 후 일괄 보고", "개인 판단으로 경보 해제"], 2, [P, P, P]),
    _mc("Q11", "경보보고", 2, "보고 내용이 불확실할 때 올바른 표현은?", ["확정 표현으로 보고", "'추정'임을 밝히고 근거와 함께 보고", "보고 보류", "다른 관측자 보고를 복사"], 2, [C, P, P]),
    _mc("Q12", "경보보고", 1, "드론 탐지 후 보고 경로로 알맞은 것은?", ["인접 부대에만 전파", "지휘계통을 따라 보고", "개인 메신저로 공유", "보고 없이 대응"], 2, [P, P, P]),
    _mc("Q13", "교전통제", 1, "AI 탐지 결과만으로 대응을 결정할 수 있는가?", ["가능하다", "인간 지휘관의 승인이 필요하다", "신뢰도 90% 이상이면 가능", "야간에만 가능"], 2, [C, C, C]),
    _mc("Q14", "교전통제", 2, "민간 피해 가능성이 확인되면?", ["그대로 진행", "즉시 보고하고 지휘관 판단을 받는다", "조치 후 보고", "무시"], 2, [P, P, F]),
    _mc("Q15", "교전통제", 3, "식별이 불확실한 표적에 대한 원칙은?", ["위협으로 간주해 즉시 대응", "식별 확인 전까지 승인 절차를 따른다", "무조건 무시", "다수결로 결정"], 2, [F, F, C]),
    _mc("Q16", "교전통제", 2, "판단 근거를 기록해야 하는 이유는?", ["형식상 필요", "사후 검토와 책임 추적", "보고 생략", "기록 불필요"], 2, [C, P, P]),
    _mc("Q17", "대드론일반", 1, "대드론 체계의 일반적 처리 순서는?", ["대응-탐지-식별", "탐지-식별-보고·판단-대응", "식별-대응-탐지", "보고-탐지-대응"], 2, [P, P, P]),
    _mc("Q18", "대드론일반", 3, "레이더만으로 조류와 드론 구분이 어려운 이유는?", ["반사 신호 크기가 비슷할 수 있어서", "레이더는 야간에 못 써서", "드론에는 금속이 없어서", "조류는 탐지되지 않아서"], 1, [C, C, C]),
    _mc("Q19", "대드론일반", 2, "전자광학(EO) 센서의 약점이 커지는 조건은?", ["맑은 주간", "안개·야간", "저속 표적", "근거리"], 2, [C, C, C]),
    _mc("Q20", "대드론일반", 2, "경보가 반복해서 오경보로 판명되면?", ["경보 장치를 끈다", "사례를 기록·보고하고 기준 재검토 요청", "무시한다", "모두 실제 위협으로 간주"], 2, [P, P, F]),
    _sa("Q21", "피아식별", 1, "아군·적·민간을 구분하는 절차를 무엇이라 하는가?", ["피아식별", "IFF"], {"표적획득": C, "적기판정": F, "경계": C}),
    _sa("Q22", "형상식별", 2, "꼬리날개 없는 삼각형 날개 형태를 무엇이라 하는가?", ["삼각익", "델타익", "delta"], {"직선익": C, "후퇴익": C, "쿼드콥터": C}),
    _sa("Q23", "경보보고", 2, "불확실한 관측을 보고할 때 붙이는 표현은?", ["추정"], {"확정": C, "확인": C, "생략": P}),
    _sa("Q24", "교전통제", 1, "AI 탐지 결과에 대해 최종 결정을 내리는 주체는?", ["지휘관"], {"AI": C, "운용병": P, "레이더": C}),
    _sa("Q25", "대드론일반", 1, "조류를 드론으로 잘못 경보하는 것을 무엇이라 하는가?", ["오경보", "오탐", "false alarm"], {"미탐": C, "재경보": C, "적기": F}),
    _sa("Q26", "피아식별", 2, "운항 정보가 확인된 여객기는 어떻게 분류하는가?", ["민간기", "민항기"], {"적기": F, "미확인기": P, "아군기": F}),
    _sa("Q27", "형상식별", 1, "회전날개 4개를 가진 드론 형태는?", ["쿼드콥터", "quadcopter"], {"헥사콥터": C, "고정익": C, "헬기": F}),
    _sa("Q28", "경보보고", 1, "보고는 무엇을 따라 올라가는가?", ["지휘계통"], {"인접부대": P, "개인메신저": P, "통신망": C}),
    _order("Q29", "경보보고", 2, "최초 보고 순서: A 관측 시각, B 방향·거리, C 추정 유형, D 관측자 조치"),
    _order("Q30", "대드론일반", 2, "처리 순서: A 탐지, B 식별, C 보고·판단, D 대응"),
    _order("Q31", "교전통제", 3, "통제 순서: A 식별 확인, B 상급 승인 요청, C 승인 수령, D 조치 기록"),
    _order("Q32", "피아식별", 3, "확인 순서: A 운항정보 대조, B 식별 신호 확인, C 미확인 분류, D 보고"),
]
QIDX = {q["qid"]: q for q in BANK}
UNITS = ["가상1중대", "가상2중대", "가상3중대"]
_CIRC = "①②③④"
PARA = {"Q21": "아군적군 구분", "Q22": "세모 날개", "Q23": "추정됨으로 표기", "Q24": "중대장(지휘관)", "Q25": "잘못된 경보",
        "Q26": "민간 여객기", "Q27": "4발 드론", "Q28": "지휘 라인"}          # 뜻은 맞지만 인정 답안 목록에 없는 표현


def make_soldiers(n_per_unit, seed):
    rng = np.random.default_rng(seed)
    out = []
    for u, unit in enumerate(UNITS):
        base = [0.3, -0.1, 0.0][u]
        for i in range(n_per_unit):
            off = rng.normal(0, 0.5, len(AREAS)) + (np.array([0, 0, 0, -0.6, 0]) if u == 1 else 0)
            out.append({"sid": f"S{len(out) + 1:03d}", "unit": unit, "theta": dict(zip(AREAS, base + rng.normal(0, 0.6) + off)),
                        "tend": dict(zip(ETYPES, rng.dirichlet([1.2, 1.2, 1.2])))})
    return out


def p_correct(theta, diff):
    return 1 / (1 + np.exp(-1.7 * (theta - (diff - 2) * 0.9 + 0.9)))


def _typo(s, rng):
    if len(s) < 2:
        return s
    i = int(rng.integers(len(s) - 1))
    return s[:i] + s[i + 1] + s[i] + s[i + 2:] if len(s) <= 3 or rng.random() < 0.5 else s[:i] + s[i + 1:]


def _fmt_mc(k, rng):
    r = rng.random()
    if r < 0.55:
        return str(k)
    return rng.choice([_CIRC[k - 1], f"{k}번", "ABCD"[k - 1], "abcd"[k - 1] + ")", f" {k} ", chr(0xFF10 + k), f"{k}."])


def _fmt_sa(s, rng):
    r = rng.random()
    if r < 0.45:
        return s
    if r < 0.6:
        return " ".join(s) if len(s) <= 4 else s[:2] + " " + s[2:]
    if r < 0.72:
        return s + rng.choice(["입니다", "이다", ".", "임"])
    if r < 0.8:
        return s.upper() if s.isascii() else "".join(chr(ord(ch) + 0xFEE0) if "!" <= ch <= "~" else ch for ch in s) + " "
    return _typo(s, rng)


def answer(q, sol, rng):
    """한 문항 응답: (원본 응답 문자열, 의도한 정답 여부, 의도한 답, 오답 유형)."""
    ok = rng.random() < p_correct(sol["theta"][q["area"]], q["diff"])
    if q["type"] == "MC":
        if ok:
            k, et = q["answer"], None
        else:
            ks = list(q["dist"]); w = np.array([sol["tend"][q["dist"][k]] for k in ks])
            k = int(rng.choice(ks, p=w / w.sum())); et = q["dist"][k]
        return _fmt_mc(k, rng), ok, str(k), et
    if q["type"] == "SA":
        if ok and rng.random() < 0.06:
            return PARA[q["qid"]], ok, PARA[q["qid"]], None
        if ok:
            s, et = str(rng.choice(q["accept"])), None
        else:
            ws = list(q["wrong"]); w = np.array([sol["tend"][q["wrong"][x]] for x in ws])
            s = str(rng.choice(ws, p=w / w.sum())); et = q["wrong"][s]
        return _fmt_sa(s, rng), ok, s, et
    if ok:
        s = "ABCD"
    elif rng.random() < 0.5:
        s = "ABCD".replace("ABCD"[rng.integers(1, 4)], "")       # 단계 하나 빠뜨림
    else:
        i = int(rng.integers(0, 3)); s = list("ABCD"); s[i], s[i + 1] = s[i + 1], s[i]; s = "".join(s)   # 순서 뒤바뀜
    raw = s if rng.random() < 0.6 else rng.choice(["-".join(s), " ".join(s), s.lower(), ",".join(s)])
    return raw, ok, s, (None if ok else P)


def make_responses(n_per_unit=40, seed=7, blank=0.02):
    """응시자 x 문항 응답표(DataFrame)를 만든다. truth_* 열은 교관 기준 정답(평가용)이다."""
    rng = np.random.default_rng(seed)
    rows = []
    for sol in make_soldiers(n_per_unit, seed):
        for q in BANK:
            raw, ok, intended, et = answer(q, sol, rng)
            if rng.random() < blank:
                raw, ok, intended, et = "", False, "", None
            rows.append({"rid": f"R{len(rows):05d}", "sid": sol["sid"], "unit": sol["unit"], "qid": q["qid"], "raw": raw,
                         "truth_correct": ok, "truth_answer": intended, "truth_etype": et})
    return pd.DataFrame(rows)
# ---------------------------------------------------------------------------

# ---- 응답 정제·파싱·채점 공통 함수 ----
TAILS = ("입니다", "이다", "임")


def norm_text(s):
    """전각->반각(NFKC), 소문자, 공백·문장부호 제거, 끝맺음 어미 제거."""
    s = unicodedata.normalize("NFKC", str(s)).strip().lower()
    s = re.sub(r"[\s\.\,\-\(\)·]", "", s)
    for t in TAILS:
        if s.endswith(t) and len(s) > len(t) + 1:
            s = s[:-len(t)]
    return s


def parse(q, raw):
    """원본 응답 -> (정제값, 상태). 상태: 정상 / 미응답 / 형식오류"""
    if raw is None or str(raw).strip() == "":
        return "", "미응답"
    s = unicodedata.normalize("NFKC", str(raw)).strip()
    if q["type"] == "MC":
        t = re.sub(r"[\s번\.\)]", "", s).lower()
        t = {"a": "1", "b": "2", "c": "3", "d": "4"}.get(t, t)
        return (t, "정상") if t in ("1", "2", "3", "4") else (t, "형식오류")
    if q["type"] == "ORDER":
        t = re.sub(r"[^A-Za-z]", "", s).upper()
        return (t, "정상") if t and set(t) <= set("ABCD") and len(set(t)) == len(t) else (t, "형식오류")
    return norm_text(s), "정상"


def grade_exact(q, val):
    if q["type"] == "MC":
        return val == str(q["answer"])
    if q["type"] == "ORDER":
        return val == q["answer"]
    return val in {norm_text(a) for a in q["accept"]}


def grade_fuzzy(q, val, cut=0.75):
    """단답형은 오탈자(글자 순서 바뀜·한 글자 누락)를 허용(유사도 cut 이상)하되, 알려진 오답과 더 가까우면 오답으로 본다."""
    if grade_exact(q, val) or q["type"] != "SA" or len(val) < 2:
        return grade_exact(q, val)
    best = max(1.0 if sorted(val) == sorted(norm_text(a)) else difflib.SequenceMatcher(None, val, norm_text(a)).ratio() for a in q["accept"])
    worst = max(difflib.SequenceMatcher(None, val, norm_text(w)).ratio() for w in q["wrong"])
    return best >= cut and best > worst


def score_of(q, correct):
    return POINTS[q["diff"]] if correct else 0


def grade_one(qid, raw):
    q = QIDX[qid]
    val, st = parse(q, raw)
    return st, (st == "정상" and grade_fuzzy(q, val))


@functools.lru_cache(maxsize=50000)
def grade_cached(qid, raw):
    """같은 (문항, 원본 응답)은 다시 계산하지 않는다(대규모 응시에서 같은 답이 반복됨)."""
    return grade_one(qid, raw)


def corrupt(df, rng, rate):
    """손상 답안 주입: 초장문, 제어문자, 존재하지 않는 문항, None."""
    df = df.copy()
    idx = rng.choice(len(df), int(len(df) * rate), replace=False)
    kinds = rng.integers(0, 4, len(idx))
    for i, k in zip(idx, kinds):
        df.at[df.index[i], "raw"] = ["가" * 20000, "\x00\x07삼각익", None, "1"][k]
        if k == 3:
            df.at[df.index[i], "qid"] = "Q99"
    return df


def run(df, fn, fail_rate=0.0, seed=0, cap=2000):
    """제출 큐에서 cap건씩 꺼내 채점. 채점 중 예외·장애가 나면 '채점 보류' 큐로 옮기고 계속 처리한다(안전 실패)."""
    rng = np.random.default_rng(seed)
    inbox, result, hold = deque(df[["rid", "qid", "raw"]].itertuples(index=False)), {}, {}
    t0, crashes, max_q = time.perf_counter(), 0, len(inbox)
    while inbox:
        batch = [inbox.popleft() for _ in range(min(cap, len(inbox)))]
        for rid, qid, raw in batch:
            try:
                if qid not in QIDX:
                    raise KeyError("문항 없음")
                if raw is not None and len(str(raw)) > 200:
                    raise ValueError("응답 길이 초과")
                if rng.random() < fail_rate:
                    raise TimeoutError("채점기 응답 없음")
                result[rid] = fn(qid, "" if raw is None else str(raw).replace("\x00", "").replace("\x07", ""))
            except Exception as e:       # 시스템은 멈추지 않고 보류 큐에 원본을 보존
                hold[rid] = type(e).__name__
    sec = time.perf_counter() - t0
    lost = len(df) - len(result) - len(hold)
    return {"건수": len(df), "처리(초)": round(sec, 2), "처리량(건/초)": int(len(df) / sec), "채점 완료": len(result),
            "보류": len(hold), "유실": lost, "다운": crashes, "최대 대기열": max_q}, hold


base = make_responses(40, seed=7)                   # 평시: 120명 x 32문항
rng = np.random.default_rng(1)
rows, holds = [], {}
for mult, fail in [(1, 0.0), (3, 0.0), (6, 0.0), (6, 0.02), (11, 0.02)]:
    df = pd.concat([base.assign(rid=base["rid"] + f"-{k}") for k in range(mult)], ignore_index=True)
    df = corrupt(df, rng, 0.003)
    grade_cached.cache_clear()
    tracemalloc.start()
    r, hold = run(df, grade_cached, fail_rate=fail, seed=mult)
    peak = tracemalloc.get_traced_memory()[1] / 1e6; tracemalloc.stop()
    rows.append({"부하": f"평시 x{mult}" + (" +장애2%" if fail else ""), **r, "메모리 최대(MB)": round(peak, 1)})
    holds[rows[-1]["부하"]] = hold
T = pd.DataFrame(rows)
print(T.drop(columns=["다운", "최대 대기열"]).to_string(index=False))
print("(위 표의 처리량은 메모리 추적을 켠 상태라 실제보다 낮게 나온다)")
w = T[T["부하"] == "평시 x6"].iloc[0]
print(f"\n동시 응시 500% 증가(평시 x6): 시스템 다운 {int(T['다운'].sum())}건, 데이터 유실 {int(T['유실'].sum())}건 "
      f"(목표 다운 0건·유실률 0%: {'달성' if T['다운'].sum() == 0 and T['유실'].sum() == 0 else '미달'})")
h = pd.Series(holds["평시 x11 +장애2%"]).value_counts()
print("보류 사유(평시 x11 +장애2%):", {k: int(v) for k, v in h.items()}, "-> 원본 보존, 장애 복구 후 재채점 또는 교관 확인")

# 병목 프로파일링과 최적화: 캐시 없음 vs 캐시
df6 = pd.concat([base.assign(rid=base["rid"] + f"-{k}") for k in range(6)], ignore_index=True)
prof = {}
t0 = time.perf_counter(); [parse(QIDX[q], r) for q, r in zip(df6["qid"], df6["raw"])]; prof["파싱"] = time.perf_counter() - t0
sa = df6[df6["qid"].map(lambda q: QIDX[q]["type"]) == "SA"]
t0 = time.perf_counter(); [grade_fuzzy(QIDX[q], parse(QIDX[q], r)[0]) for q, r in zip(sa["qid"], sa["raw"])]; prof["단답 유사도 채점"] = time.perf_counter() - t0
print("\n[병목 프로파일] " + ", ".join(f"{k} {v * 1000:.0f} ms" for k, v in prof.items()) + f" (단답형은 전체의 {len(sa) / len(df6):.0%})")
r0, _ = run(df6, grade_one)
grade_cached.cache_clear(); r1, _ = run(df6, grade_cached)
print(f"최적화 전 {r0['처리량(건/초)']:,}건/초 -> 캐시 적용 후 {r1['처리량(건/초)']:,}건/초 ({r1['처리량(건/초)'] / r0['처리량(건/초)']:.1f}배)")
print(f"기능 축소 운용: 장애율이 5%를 넘으면 단답형 유사도 채점을 끄고 정확 일치만 채점, 나머지는 교관 검토로 넘긴다.")

plt.figure(figsize=(8, 4))
plt.bar(T["부하"].str.replace("평시", "base").str.replace("+장애2%", "+fail2%"), T["처리량(건/초)"])
plt.ylabel("responses / sec"); plt.title("Grading throughput under load (cached)"); plt.xticks(rotation=15)
plt.tight_layout(); plt.show()


<details>
<summary><b>💡 [참고] 미션 08 예상 실행 결과 (Output)</b></summary>

```text
          부하    건수  처리(초)  처리량(건/초)  채점 완료  보류  유실  메모리 최대(MB)
       평시 x1  3840   0.11     33625   3834   6   0         0.5
       평시 x3 11520   0.19     59703  11501  19   0         1.1
       평시 x6 23040   0.27     86855  23004  36   0         2.0
 평시 x6 +장애2% 23040   0.24     96047  22558 482   0         2.0
평시 x11 +장애2% 42240   0.46     91783  41326 914   0         3.6
(위 표의 처리량은 메모리 추적을 켠 상태라 실제보다 낮게 나온다)

동시 응시 500% 증가(평시 x6): 시스템 다운 0건, 데이터 유실 0건 (목표 다운 0건·유실률 0%: 달성)
보류 사유(평시 x11 +장애2%): {'TimeoutError': 857, 'ValueError': 30, 'KeyError': 27} -> 원본 보존, 장애 복구 후 재채점 또는 교관 확인

[병목 프로파일] 파싱 52 ms, 단답 유사도 채점 150 ms (단답형은 전체의 25%)
최적화 전 89,083건/초 -> 캐시 적용 후 498,682건/초 (5.6배)
기능 축소 운용: 장애율이 5%를 넘으면 단답형 유사도 채점을 끄고 정확 일치만 채점, 나머지는 교관 검토로 넘긴다.
```
</details>
